# CLIM-715 Course Project
## Numerical Damping of the Diurnal Ground Heat Flux over Urban Substrates

**A Comparison of Explicit, Implicit, and Crank–Nicolson Schemes Coupled to a Prognostic Surface Energy Balance**

Submitted by **Shammunul Islam**
Department of Atmospheric, Oceanic and Earth Sciences
George Mason University

---

This notebook reproduces every figure and table in the project report. It is organised as:

1. **Setup** — Google Drive mount, imports, plot styling.
2. **Solver implementation** — α-weighted θ-method on a non-uniform grid with depth-varying λs(z) and Cs(z).
3. **Figure 1** — von Neumann amplification factors (analytical) for FTCS, BTCS, CN.
4. **Test 1 (Figure 2)** — Damping-depth verification on a uniform sandy-loam column. Sinusoidal Dirichlet upper BC, zero-flux Neumann lower BC, comparison against the analytical solution Ts(z,t) = T̄ + A exp(−z/d) cos(ωt − z/d).
5. **Test 2 (Figures 3, 4, 5)** — Prognostic surface energy balance with Newton iteration on three substrate columns (asphalt road, concrete roof with insulation, bare soil).
6. **Cross-substrate metrics** — diurnal G amplitude ratio, surface-T RMSE, daily heat storage.

The numerical schemes and analysis follow the lecture material directly: the parabolic PDE classification (Notes #2), staggered grids and harmonic-mean conductivity at half-levels (Notes #8), the von Neumann analysis (Lecture 6 Slides 13–17), the order-of-operations splitting between the soil column and the SEB (Notes #10 Section 8), and the resolution-vs-stability discussion (Notes #11 Misconception #1).

## 1. Setup

### 1a. Mount Google Drive

All figures will be saved to a `Project_Ground_Flux` subfolder under the course folder on Drive.

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/My Drive/Numerical Simulation for Weather and Climate"
OUT = os.path.join(BASE, "Project_Ground_Flux")
os.makedirs(OUT, exist_ok=True)
print(f"Output dir: {OUT}")

### 1b. Imports and plot styling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

%matplotlib inline

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['axes.facecolor'] = '#fafafa'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# scheme colours (matched between figures)
C_FTCS = '#7B61FF'
C_BTCS = '#00B8A9'
C_CN   = '#F08A24'
C_REF  = '#666666'
C_ANA  = '#000000'
print('imports OK')

## 2. Solver implementation

We solve the 1D vertical heat-conduction equation in the substrate

$$C_s(z) \frac{\partial T_s}{\partial t} = \frac{\partial}{\partial z} \left( \lambda_s(z) \frac{\partial T_s}{\partial z} \right)$$

on a (non-uniform) vertical grid with depth-varying thermal conductivity λs(z) and volumetric heat capacity Cs(z). The conductivity form is the right discrete formulation for layered substrates because both λ and C change by an order of magnitude across material interfaces (e.g. concrete/insulation, factor of 37 in λ).

### 2a. Grid construction

Cell centres at zⱼ; cell thicknesses Δzⱼ; centre-to-centre spacings Δzc_{j+1/2} = z_{j+1} − zⱼ. We use either a uniform grid (Test 1) or a stretched grid with Δz₁ = 0.5–1 cm at the surface and ~30 cm at z = 2 m (Test 2).

In [ ]:
def make_grid(z_top=2.0, n_levels=20, stretch=1.25, dz1=0.01):
    """Stretched grid downward from the surface."""
    dzs = dz1 * stretch ** np.arange(n_levels)
    if np.sum(dzs) > z_top:
        cumulative = np.cumsum(dzs)
        n_levels = int(np.searchsorted(cumulative, z_top)) + 1
        dzs = dzs[:n_levels]
    dzs[-1] = z_top - np.sum(dzs[:-1])
    z = np.zeros(n_levels)
    z[0] = dzs[0] / 2.0
    for j in range(1, n_levels):
        z[j] = z[j - 1] + (dzs[j - 1] + dzs[j]) / 2.0
    dzc = np.diff(z)
    return z, dzs, dzc


def make_uniform_grid(z_top=2.0, dz=0.01):
    """Uniform grid for the verification test."""
    n_levels = int(round(z_top / dz)) + 1
    z = np.arange(n_levels) * dz
    dzs = np.full(n_levels, dz)
    dzs[0] = dz / 2.0
    dzs[-1] = dz / 2.0
    dzc = np.full(n_levels - 1, dz)
    return z, dzs, dzc


def assign_layered_props(z, layers):
    """Given layers = [(z_bottom, lambda, C), ...], assign at cell centres
    and harmonic-mean λ at half-levels (preserves heat flux across interfaces)."""
    N = len(z)
    lam = np.zeros(N); C = np.zeros(N)
    for j in range(N):
        for (zb, lj, cj) in layers:
            if z[j] <= zb:
                lam[j] = lj; C[j] = cj
                break
        else:
            lam[j] = layers[-1][1]; C[j] = layers[-1][2]
    lam_half = 2.0 / (1.0 / lam[:-1] + 1.0 / lam[1:])
    return lam, C, lam_half

print('grid functions OK')

### 2b. The α-weighted θ-method

A single solver routine handles all three schemes through the parameter α:

- α = 0 → FTCS (forward Euler)
- α = 1 → BTCS (backward Euler)
- α = 1/2 → Crank–Nicolson

The discrete update at cell j is

$$C_j \, \Delta z_j \, \frac{T_j^{n+1} - T_j^n}{\Delta t} = \alpha [G_{j-1/2} - G_{j+1/2}]^{n+1} + (1-\alpha) [G_{j-1/2} - G_{j+1/2}]^n$$

with G_{j+1/2} = λ_{j+1/2} (Tⱼ − T_{j+1}) / Δzc_{j+1/2}. The implicit cases produce a tridiagonal system A T_{n+1} = b solved by SciPy's banded solver in O(N) per step.

In [ ]:
def step_alpha(T, dt, dzs, dzc, lam_half, C, alpha,
               T_top=None, lower_bc='neumann'):
    """Advance T by dt using the alpha-weighted theta-method.

    alpha = 0   : FTCS (explicit)
    alpha = 1   : BTCS
    alpha = 0.5 : Crank-Nicolson

    Top BC: Dirichlet, T[0] = T_top.
    Bottom BC: zero-flux Neumann (default) or Dirichlet (hold T[-1]).
    """
    N = len(T)

    if alpha == 0.0:
        # explicit forward Euler step
        T_new = T.copy()
        for j in range(1, N - 1):
            flux_up = lam_half[j]     * (T[j + 1] - T[j])     / dzc[j]
            flux_dn = lam_half[j - 1] * (T[j]     - T[j - 1]) / dzc[j - 1]
            T_new[j] = T[j] + (dt / (C[j] * dzs[j])) * (flux_up - flux_dn)
        if T_top is not None:
            T_new[0] = T_top
        if lower_bc == 'neumann':
            T_new[-1] = T_new[-2]
        else:
            T_new[-1] = T[-1]
        return T_new

    # build banded system A T^{n+1} = b for alpha > 0
    diag_main  = np.ones(N)
    diag_upper = np.zeros(N)
    diag_lower = np.zeros(N)
    rhs = T.copy()

    for j in range(1, N - 1):
        a = dt * lam_half[j - 1] / (C[j] * dzs[j] * dzc[j - 1])
        b = dt * lam_half[j]     / (C[j] * dzs[j] * dzc[j])
        diag_lower[j - 1] = -alpha * a
        diag_main[j]      = 1.0 + alpha * (a + b)
        diag_upper[j + 1] = -alpha * b
        rhs[j] = (T[j]
                  + (1 - alpha) * a * (T[j - 1] - T[j])
                  + (1 - alpha) * b * (T[j + 1] - T[j]))

    # boundary rows
    if T_top is None:
        T_top = T[0]
    diag_main[0] = 1.0
    diag_upper[1] = 0.0
    rhs[0] = T_top

    if lower_bc == 'neumann':
        diag_main[N - 1] = 1.0
        diag_lower[N - 2] = -1.0
        rhs[N - 1] = 0.0
    else:
        diag_main[N - 1] = 1.0
        diag_lower[N - 2] = 0.0
        rhs[N - 1] = T[N - 1]

    ab = np.zeros((3, N))
    ab[0, :] = diag_upper
    ab[1, :] = diag_main
    ab[2, :] = diag_lower
    return solve_banded((1, 1), ab, rhs)

print('step_alpha OK')

### 2c. von Neumann amplification factors

Substituting a Fourier mode Tⱼⁿ = Aⁿ exp(i k j Δz) into each scheme on a uniform grid with constant κs = λs/Cs gives the standard amplification factors (Lecture 6 Slides 13–17):

$$A_{\text{FTCS}}(\nu, k\Delta z) = 1 - 2\nu (1 - \cos k\Delta z)$$

$$A_{\text{BTCS}}(\nu, k\Delta z) = \frac{1}{1 + 2\nu (1 - \cos k\Delta z)}$$

$$A_{\text{CN}}(\nu, k\Delta z) = \frac{1 - \nu (1 - \cos k\Delta z)}{1 + \nu (1 - \cos k\Delta z)}$$

with ν = κs Δt / Δz². At the worst-case wave kΔz = π: FTCS is conditionally stable (bound ν ≤ 1/2); BTCS satisfies |A| < 1 for all positive ν; CN satisfies |A| ≤ 1 always but A_CN(ν, π) → −1 as ν → ∞.

In [ ]:
def amp_ftcs(nu, kdz):
    return 1.0 - 2.0 * nu * (1.0 - np.cos(kdz))

def amp_btcs(nu, kdz):
    return 1.0 / (1.0 + 2.0 * nu * (1.0 - np.cos(kdz)))

def amp_cn(nu, kdz):
    h = nu * (1.0 - np.cos(kdz))
    return (1.0 - h) / (1.0 + h)

# numerical values at the 2*dz wave
print(f"{'nu':>8} {'A_FTCS':>12} {'A_BTCS':>12} {'A_CN':>12}")
print('-' * 46)
for nu in [0.25, 0.5, 1.0, 5.0, 50.0]:
    print(f"{nu:>8.2f} {amp_ftcs(nu, np.pi):>+12.4f} "
          f"{amp_btcs(nu, np.pi):>+12.4f} {amp_cn(nu, np.pi):>+12.4f}")

## 3. Figure 1 — Amplification factors

We plot |A_k(ν, kΔz)| for ν = 0.25, 0.5, 1.0, 5.0 across the resolved wavenumber range [0, π].

**What to look for:**
- **FTCS:** stable curves stay below |A| = 1; unstable curves exceed it geometrically.
- **BTCS:** monotonically decreasing; strict damping for all ν > 0.
- **CN:** bounded by |A| = 1 but the curve for ν = 5 rises again at high wavenumber to |A| ≈ 0.82, illustrating weak damping of the 2Δz mode at large ν.

In [ ]:
kdz = np.linspace(0, np.pi, 200)
nu_values = [0.25, 0.5, 1.0, 5.0]
nu_colors = ['#5B8BD8', '#F08A24', '#7B61FF', '#D14545']

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
titles = ['FTCS (explicit)', 'BTCS (fully implicit)', 'Crank-Nicolson']
funcs = [amp_ftcs, amp_btcs, amp_cn]
ylims = [(0, 4.0), (0, 1.1), (0, 1.1)]

for ax, title, fn, ylim in zip(axes, titles, funcs, ylims):
    for nu, col in zip(nu_values, nu_colors):
        ax.plot(kdz, np.abs(fn(nu, kdz)), label=fr'$\nu={nu}$',
                linewidth=1.7, color=col)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=0.9, alpha=0.7)
    ax.set_xlabel(r'$k\,\Delta z$')
    ax.set_ylabel(r'$|A_k|$')
    ax.set_title(title)
    ax.set_xlim(0, np.pi)
    ax.set_xticks([0, np.pi / 2, np.pi])
    ax.set_xticklabels(['0', r'$\pi/2$', r'$\pi$'])
    ax.set_ylim(*ylim)
    ax.legend(loc='best', frameon=False)
    ax.grid(True, alpha=0.3)

axes[0].annotate(r'$\nu=5$ off-scale (max $\approx 19$)',
                 xy=(np.pi*0.95, 3.6),
                 xytext=(np.pi*0.4, 3.6), ha='left', fontsize=8,
                 color='#D14545',
                 arrowprops=dict(arrowstyle='->', color='#D14545', lw=0.8))

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig1_amplification.png'), bbox_inches='tight')
plt.show()

## 4. Test 1 — Damping-depth verification (Figure 2)

The diurnal damping-depth problem in a semi-infinite uniform medium with sinusoidal Dirichlet forcing Ts(0, t) = T̄ + A cos(ωt) has the closed-form solution

$$T_s(z, t) = \bar T + A \, e^{-z/d} \cos(\omega t - z/d), \qquad d = \sqrt{\frac{2 \kappa}{\omega}}.$$

The corresponding analytical surface ground heat flux is

$$G(0, t) = \lambda \frac{A}{d} \sqrt{2} \, \cos(\omega t + \pi/4),$$

which leads the surface temperature by π/4 (3 hours).

We run the three schemes on a uniform sandy-loam column (λ = 0.30 W m⁻¹ K⁻¹, C = 1.3 × 10⁶ J m⁻³ K⁻¹, κ = 2.31 × 10⁻⁷ m² s⁻¹, d = 7.97 cm; depth 2 m, dz = 1 cm, N = 201 cells), six configurations:

- FTCS at ν = 0.4 (stable) and ν = 0.6 (unstable, expected to blow up);
- BTCS at Δt = 300 s (ν ≈ 0.69) and Δt = 900 s (ν ≈ 2.08);
- CN at Δt = 300 s and Δt = 900 s.

Each integration runs for 5 days; we compare day 5 against the analytical solution.

### 4a. Run all six configurations

In [ ]:
# uniform sandy loam
lam_s = 0.30           # W/m/K
C_s = 1.3e6            # J/m^3/K
kappa = lam_s / C_s    # m^2/s, ~ 2.31e-7
omega = 2.0 * np.pi / 86400.0
d = np.sqrt(2.0 * kappa / omega)
print(f'damping depth d = {d*100:.2f} cm')

T0 = 290.0
A_amp = 10.0

dz = 0.01
z, dzs, dzc = make_uniform_grid(z_top=2.0, dz=dz)
N = len(z)
lam_arr = np.full(N, lam_s)
C_arr = np.full(N, C_s)
lam_half = np.full(N - 1, lam_s)

dt_crit = 0.5 * dz**2 / kappa
print(f'FTCS critical dt = {dt_crit:.1f} s')


def analytical(z_arr, t):
    return T0 + A_amp * np.exp(-z_arr / d) * np.cos(omega * t - z_arr / d)


def G_analytical(t):
    return lam_s * (A_amp / d) * np.sqrt(2.0) * np.cos(omega * t + np.pi / 4.0)


configs = [
    ('FTCS (nu=0.4)',         0.0, 0.4 * dt_crit / 0.5),
    ('FTCS (nu=0.6, unstable)', 0.0, 0.6 * dt_crit / 0.5),
    ('BTCS dt=300s',          1.0, 300.0),
    ('CN   dt=300s',          0.5, 300.0),
    ('BTCS dt=900s',          1.0, 900.0),
    ('CN   dt=900s',          0.5, 900.0),
]

n_days = 5
t_end = n_days * 86400.0
test1 = []

for name, alpha, dt in configs:
    n_steps = int(round(t_end / dt))
    dt = t_end / n_steps
    nu_actual = (lam_s / C_s) * dt / dz**2
    T = np.full(N, T0)
    save_every = max(1, n_steps // (24 * n_days))
    ts = [0.0]; Ts_surface = [T[0]]; T_at_10cm = [T[10]]; G_surface = [0.0]
    blew_up = False
    for n in range(1, n_steps + 1):
        t_now = n * dt
        T_top = T0 + A_amp * np.cos(omega * t_now)
        T = step_alpha(T, dt, dzs, dzc, lam_half, C_arr, alpha,
                       T_top=T_top, lower_bc='neumann')
        if not np.all(np.isfinite(T)) or np.max(np.abs(T - T0)) > 1e6:
            blew_up = True
            print(f'  {name} BLEW UP at step {n}, t = {t_now:.1f} s')
            break
        if n % save_every == 0:
            ts.append(t_now); Ts_surface.append(T[0])
            T_at_10cm.append(T[10])
            G_surface.append(-lam_half[0] * (T[1] - T[0]) / dzc[0])
    test1.append({
        'name': name, 'alpha': alpha, 'dt': dt, 'nu': nu_actual,
        'ts': np.array(ts), 'Ts_surface': np.array(Ts_surface),
        'T_at_10cm': np.array(T_at_10cm),
        'G_surface': np.array(G_surface),
        'blew_up': blew_up,
        'T_final': T,
    })
    print(f'  {name:30s}  dt={dt:7.1f} s  nu={nu_actual:6.2f}  steps={n_steps:6d}'
          f'  ' + ('BLEW UP' if blew_up else 'OK'))

### 4b. Plot Figure 2 — three-panel verification

(a) Vertical Ts profile at t = 24 h on day 5, top 100 cm.

(b) Surface Ts(t) over day 5; numerical curves should overlay the analytical (Ts is enforced as Dirichlet, so the surface match is exact up to floating-point error).

(c) Surface ground heat flux G(t) on day 5; the three schemes should match the analytical π/4 phase lead and amplitude. The ~3 W m⁻² residual is the spatial-discretization error of the finite-difference first derivative, independent of the time-stepping scheme.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))

t_day5_start = 4 * 86400.0
t_day5_end = 5 * 86400.0
t_noon = 4 * 86400.0  # cos(omega*4*86400) = 1, peak surface forcing
z_arr = np.linspace(0, 1.0, 500)

# (a) profile at peak surface forcing
ax = axes[0]
ax.plot(analytical(z_arr, t_noon), z_arr * 100, color=C_ANA, linewidth=2.0,
        label='Analytical')
# we don't store the column profile every step; recompute on request
# from the final state (which is at end of day 5, omega*5*86400 = 10 pi -> cos=1, peak again)
# So T_final IS at t_noon equivalent (mod 24 h)
for r in test1:
    if r['blew_up']:
        continue
    if '900s' in r['name'] or '0.4' in r['name']:
        ax.plot(r['T_final'], z * 100, linewidth=1.4, alpha=0.85,
                label=r['name'])
ax.invert_yaxis()
ax.set_xlabel(r'$T_s$ (K)')
ax.set_ylabel('Depth (cm)')
ax.set_title('(a) Profile at $t$ = 24 h (peak surface forcing)')
ax.legend(loc='lower right', frameon=False, fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(100, 0)

# (b) surface T over day 5
ax = axes[1]
t_anal = np.linspace(t_day5_start, t_day5_end, 500)
ax.plot((t_anal - t_day5_start) / 3600.0,
        analytical(0.0, t_anal), color=C_ANA, linewidth=2.0, label='Analytical')
for r in test1:
    if r['blew_up']:
        continue
    if not ('900s' in r['name'] or '0.4' in r['name']):
        continue
    mask = (r['ts'] >= t_day5_start) & (r['ts'] <= t_day5_end)
    col = C_FTCS if r['alpha'] == 0 else (C_BTCS if r['alpha'] == 1 else C_CN)
    ax.plot((r['ts'][mask] - t_day5_start) / 3600.0,
            r['Ts_surface'][mask], color=col, linewidth=1.4,
            label=r['name'], alpha=0.85)
ax.set_xlabel('Hours into day 5')
ax.set_ylabel(r'$T_s(0, t)$ (K)')
ax.set_title(r'(b) Surface temperature, day 5')
ax.legend(loc='upper right', frameon=False, fontsize=8)
ax.grid(True, alpha=0.3)

# (c) surface G over day 5
ax = axes[2]
ax.plot((t_anal - t_day5_start) / 3600.0,
        G_analytical(t_anal), color=C_ANA, linewidth=2.0, label='Analytical')
for r in test1:
    if r['blew_up']:
        continue
    if not ('900s' in r['name'] or '0.4' in r['name']):
        continue
    mask = (r['ts'] >= t_day5_start) & (r['ts'] <= t_day5_end)
    col = C_FTCS if r['alpha'] == 0 else (C_BTCS if r['alpha'] == 1 else C_CN)
    ax.plot((r['ts'][mask] - t_day5_start) / 3600.0,
            r['G_surface'][mask], color=col, linewidth=1.4,
            label=r['name'], alpha=0.85)
ax.set_xlabel('Hours into day 5')
ax.set_ylabel(r'$G$ at $z = 0$ (W/m$^2$)')
ax.set_title(r'(c) Surface ground heat flux, day 5')
ax.legend(loc='upper right', frameon=False, fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2_damping_depth.png'), bbox_inches='tight')
plt.show()

### 4c. Quantitative day-5 errors

Surface T is enforced as Dirichlet, so the surface error is structurally zero. We diagnose the scheme by:
- **RMSE T at z = 10 cm** — amplitude and phase encode the scheme's damping/dispersion behaviour.
- **RMSE G at the surface** — involves the discretised first derivative; the time-step-dependent component reveals scheme accuracy.

In [ ]:
def compute_lag(signal, reference, times):
    s = signal - np.mean(signal); r = reference - np.mean(reference)
    n = len(s)
    if n < 4:
        return np.nan
    corr = np.correlate(s, r, mode='full')
    lags = np.arange(-n + 1, n)
    dt_s = times[1] - times[0]
    max_lag = int(round(3 * 3600.0 / dt_s))
    centre = n - 1
    sl = corr[centre - max_lag: centre + max_lag + 1]
    sl_lags = lags[centre - max_lag: centre + max_lag + 1]
    best = sl_lags[int(np.argmax(sl))]
    return best * dt_s / 60.0


print('Test 1 day-5 errors:')
print(f"{'config':30s} {'dt(s)':>8} {'nu':>7} "
      f"{'RMSE T(10cm)':>14} {'RMSE G':>10} {'lag_G(min)':>11}")
print('-' * 78)
for r in test1:
    if r['blew_up']:
        print(f"{r['name']:30s} {r['dt']:>8.1f} {r['nu']:>7.2f}   BLEW UP")
        continue
    mask = (r['ts'] >= 4 * 86400.0) & (r['ts'] <= 5 * 86400.0)
    t_arr = r['ts'][mask]
    T10 = r['T_at_10cm'][mask]; G = r['G_surface'][mask]
    T10_anal = analytical(z[10], t_arr)
    G_anal = G_analytical(t_arr)
    rmse_T = float(np.sqrt(np.mean((T10 - T10_anal)**2)))
    rmse_G = float(np.sqrt(np.mean((G - G_anal)**2)))
    lag = compute_lag(G, G_anal, t_arr)
    print(f"{r['name']:30s} {r['dt']:>8.1f} {r['nu']:>7.2f} "
          f"{rmse_T:>14.4f} {rmse_G:>10.2f} {lag:>11.1f}")

**Key observations:**

1. **FTCS at ν = 0.6** — only 20 % above the bound — destroys the solution within ~50 steps. The bound is strict; violating it is not "less accurate", it is unbounded.

2. **BTCS error grows linearly with Δt:** 0.024 K (300 s) → 0.062 K (900 s), ratio 2.6. Consistent with first-order temporal accuracy.

3. **CN error is essentially flat:** 0.005 K (300 s) → 0.006 K (900 s). Consistent with second-order temporal accuracy. CN at Δt = 900 s (ν = 2.08) is *more* accurate than FTCS at Δt = 173 s (ν = 0.4), with ~5× fewer steps.

4. The 3 W m⁻² RMS in G is the spatial-discretization residual on the 1 cm grid, scheme-independent.

## 5. Test 2 — Prognostic surface energy balance (Figures 3, 4, 5)

We now drop the prescribed Dirichlet upper boundary and solve a fully prognostic surface energy balance at each step:

$$R_n(T_s^0) - H(T_s^0) - LE(T_s^0) - G(T_s^0) = 0$$

where each flux depends on the surface temperature $T_s^0$. The radiation term $R_n = (1-\alpha) S\!\downarrow + \varepsilon L\!\downarrow - \varepsilon \sigma (T_s^0)^4$ is nonlinear in $T_s^0$, so we solve the residual by Newton iteration. The order of operations follows Notes #10 Section 8: at each step we (i) advance the soil column with the current $T_s^0$ as Dirichlet, then (ii) update $T_s^0$ from the SEB with the new T₁.

**Three substrates** are integrated in parallel:

- **Asphalt road** — asphalt 0–5 cm, λ = 0.75; aggregate 5–25 cm, λ = 1.40; dry soil 25–100 cm, λ = 0.30; subsoil 100–200 cm, λ = 0.50.
- **Concrete roof** — concrete deck 0–10 cm, λ = 1.50; rigid insulation 10–20 cm, λ = 0.04; drywall/wood 20–200 cm, λ = 0.15.
- **Bare soil** — uniform sandy loam λ = 0.30.

### 5a. Substrate definitions and synthetic forcing

In [ ]:
SUBSTRATES = {
    'asphalt_road': [
        (0.05, 0.75,  2.0e6),
        (0.25, 1.40,  2.4e6),
        (1.00, 0.30,  1.3e6),
        (2.00, 0.50,  1.8e6),
    ],
    'concrete_roof': [
        (0.10, 1.50,  2.1e6),
        (0.20, 0.04,  0.08e6),
        (2.00, 0.15,  1.5e6),
    ],
    'bare_soil': [
        (2.00, 0.30,  1.3e6),
    ],
}

SUBSTRATE_NAMES = {
    'asphalt_road':  'Asphalt road',
    'concrete_roof': 'Concrete roof',
    'bare_soil':     'Bare soil',
}

# physical / forcing constants
SIGMA_SB = 5.670374419e-8
ALBEDO   = {'asphalt_road': 0.10, 'concrete_roof': 0.30, 'bare_soil': 0.20}
EMIS     = {'asphalt_road': 0.95, 'concrete_roof': 0.92, 'bare_soil': 0.95}
C_H = 5.0e-3
RHO_AIR = 1.2
CP_AIR = 1005.0


def S_down(t):
    omega = 2.0 * np.pi / 86400.0
    val = 1000.0 * np.cos(omega * (t - 12 * 3600.0))
    return np.maximum(val, 0.0)


def L_down_func(t):
    omega = 2.0 * np.pi / 86400.0
    return 350.0 + 20.0 * np.cos(omega * (t - 14 * 3600.0))


def T_air(t):
    omega = 2.0 * np.pi / 86400.0
    return 292.5 + 7.5 * np.cos(omega * (t - 14 * 3600.0))


def U_wind(_t):
    return 3.0

print('substrates and forcing OK')

### 5b. Newton iteration on the surface energy balance

The SEB residual is a transcendental equation in $T_s^0$ because of the $(T_s^0)^4$ term in $R_n$. Newton iteration with analytical derivative converges in 3–5 iterations from a warm start.

In [ ]:
def seb_residual(Ts0, T_int, dzc0, lam_h0, t, surface_key):
    """SEB residual at the surface. Positive = excess energy at surface."""
    albedo = ALBEDO[surface_key]
    emis = EMIS[surface_key]
    Sd = S_down(t); Ld = L_down_func(t); Ta = T_air(t); U = U_wind(t)
    r_a = 1.0 / (C_H * U)

    R_n = (1.0 - albedo) * Sd + emis * Ld - emis * SIGMA_SB * Ts0**4
    H = RHO_AIR * CP_AIR * (Ts0 - Ta) / r_a
    LE = 0.0
    G = lam_h0 * (Ts0 - T_int) / dzc0
    return R_n - H - LE - G


def seb_dresidual_dT(Ts0, dzc0, lam_h0, surface_key):
    emis = EMIS[surface_key]
    dRn = -4.0 * emis * SIGMA_SB * Ts0**3
    U = U_wind(0.0); r_a = 1.0 / (C_H * U)
    dH = RHO_AIR * CP_AIR / r_a
    dG = lam_h0 / dzc0
    return dRn - dH - dG


def solve_surface_energy_balance(T_int, dzc0, lam_h0, t, surface_key,
                                  Ts0_init, tol=1e-4, max_iter=20):
    Ts0 = Ts0_init
    for _ in range(max_iter):
        f = seb_residual(Ts0, T_int, dzc0, lam_h0, t, surface_key)
        df = seb_dresidual_dT(Ts0, dzc0, lam_h0, surface_key)
        Ts0 += -f / df
        if abs(f / df) < tol:
            return Ts0, True
    return Ts0, False

print('Newton SEB OK')

### 5c. Initialization (Option B — analytical damping-depth profile)

Cold-starting from a uniform $T_s$ requires several days of spin-up. Instead, we initialize each column at midnight with the analytical damping-depth solution (using the topmost layer's $\kappa$), then refine $T_s^0$ with one Newton SEB step. This places each column close to diurnal equilibrium and lets us run only 2 days, with diagnostics on day 2.

In [ ]:
def initial_profile(z, T0, A_amp, omega, kappa_eff, t0=0.0):
    d = np.sqrt(2.0 * kappa_eff / omega)
    return T0 + A_amp * np.exp(-z / d) * np.cos(omega * t0 - z / d)


def run_seb_column(surface_key, alpha, dt, n_days=2, store_every=1):
    layers = SUBSTRATES[surface_key]
    z_top = 2.0
    if surface_key == 'concrete_roof':
        z, dzs, dzc = make_grid(z_top=z_top, n_levels=30, stretch=1.18, dz1=0.005)
    elif surface_key == 'asphalt_road':
        z, dzs, dzc = make_grid(z_top=z_top, n_levels=28, stretch=1.20, dz1=0.005)
    else:
        z, dzs, dzc = make_grid(z_top=z_top, n_levels=24, stretch=1.25, dz1=0.01)

    lam, C, lam_half = assign_layered_props(z, layers)
    omega = 2.0 * np.pi / 86400.0
    kappa_top = lam[0] / C[0]
    T_mean = 292.5; A0 = 7.5
    T = initial_profile(z, T_mean, A0, omega, kappa_top, t0=0.0)
    Ts0, _ = solve_surface_energy_balance(T[1], dzc[0], lam_half[0], 0.0,
                                          surface_key, T[0])
    T[0] = Ts0

    t_end = n_days * 86400.0
    n_steps = int(round(t_end / dt))
    dt_use = t_end / n_steps

    times = [0.0]; Ts0_arr = [T[0]]
    G_arr = [lam_half[0] * (T[0] - T[1]) / dzc[0]]
    profiles = [T.copy()]

    blew_up = False
    for n in range(1, n_steps + 1):
        t_now = n * dt_use
        T = step_alpha(T, dt_use, dzs, dzc, lam_half, C, alpha,
                       T_top=T[0], lower_bc='neumann')
        if not np.all(np.isfinite(T)) or np.max(np.abs(T - T_mean)) > 1e5:
            blew_up = True
            print(f'    {surface_key} alpha={alpha} dt={dt_use:.1f}s BLEW UP at step {n}')
            break
        Ts0, _ = solve_surface_energy_balance(T[1], dzc[0], lam_half[0],
                                              t_now, surface_key, T[0])
        T[0] = Ts0
        if n % store_every == 0 or n == n_steps:
            times.append(t_now); Ts0_arr.append(T[0])
            G_arr.append(lam_half[0] * (T[0] - T[1]) / dzc[0])
            profiles.append(T.copy())

    return {
        'surface': surface_key, 'alpha': alpha, 'dt': dt_use,
        'z': z, 'dzs': dzs, 'dzc': dzc,
        'lam': lam, 'C': C, 'lam_half': lam_half,
        'times': np.array(times),
        'Ts0': np.array(Ts0_arr),
        'G': np.array(G_arr),
        'profiles': np.array(profiles),
        'blew_up': blew_up, 'n_steps': n_steps,
    }

print('run_seb_column OK')

### 5d. Run the 27-cell experiment matrix

Three substrates × three schemes × three time steps = 27 runs.

**Time steps:**
- **Δt = 15 s** — below the FTCS stability bound on all three substrates (used as the high-resolution reference).
- **Δt = 60 s** — typical mesoscale-model time step. Exceeds the FTCS bound on the concrete roof (where the 0.5 cm concrete top cell has Δt_crit ≈ 17 s).
- **Δt = 600 s** — typical regional/climate-model time step. Far beyond the FTCS bound on all three substrates.

In [ ]:
surfaces = ['asphalt_road', 'concrete_roof', 'bare_soil']
schemes = [('FTCS', 0.0), ('BTCS', 1.0), ('CN', 0.5)]
dts = [15.0, 60.0, 600.0]

test2 = {}
for surface in surfaces:
    for sname, alpha in schemes:
        for dt in dts:
            print(f'  {surface:14s} | {sname} | dt={dt:5.1f}s ...')
            r = run_seb_column(surface, alpha, dt, n_days=2,
                               store_every=max(1, int(round(60.0 / dt))))
            test2[(surface, sname, dt)] = r
print('done')

### 5e. Figure 3 — Surface temperature at dt = 15 s (schemes agree)

At small Δt the schemes are visually indistinguishable on every substrate. Substrate-dependent differences in peak surface T (asphalt 50 °C, concrete roof 45 °C, bare soil 51 °C) are governed entirely by substrate properties and albedo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
schemes_to_plot = [('FTCS', 0.0, C_FTCS), ('BTCS', 1.0, C_BTCS), ('CN', 0.5, C_CN)]

for ax, surface in zip(axes, surfaces):
    for sname, alpha, col in schemes_to_plot:
        r = test2[(surface, sname, 15.0)]
        if r['blew_up']:
            continue
        t_day2 = r['times'] >= 86400.0
        t_h = (r['times'][t_day2] - 86400.0) / 3600.0
        ax.plot(t_h, r['Ts0'][t_day2] - 273.15, color=col, linewidth=1.6,
                label=sname, alpha=0.9)
    tt = np.linspace(0, 86400, 200)
    ax.plot(np.linspace(0, 24, 200),
            T_air(86400.0 + tt) - 273.15,
            color='grey', linewidth=1.0, linestyle=':', label=r'$T_a$')
    ax.set_xlabel('Hour of day')
    ax.set_title(SUBSTRATE_NAMES[surface])
    ax.set_xlim(0, 24); ax.set_xticks([0, 6, 12, 18, 24])
    ax.legend(loc='upper right', frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r'$T_s(0, t)$ ($^\circ$C)')

plt.suptitle(r'Day-2 surface temperature, $\Delta t = 15$ s '
             r'(schemes agree at small $\Delta t$)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3_seb_dt15.png'), bbox_inches='tight')
plt.show()

### 5f. Figure 4 — Diurnal G at dt = 600 s (key result)

This is the central diagnostic of the project. At Δt = 600 s, FTCS is unstable on the asphalt road and concrete roof. The implicit schemes complete every run, but with substrate-dependent error: BTCS overshoots the daytime peak by 40 % on asphalt and concrete; CN halves the error.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)

for ax, surface in zip(axes, surfaces):
    r_ref = test2[(surface, 'FTCS', 15.0)]
    if not r_ref['blew_up']:
        t_day2 = r_ref['times'] >= 86400.0
        t_h = (r_ref['times'][t_day2] - 86400.0) / 3600.0
        ax.plot(t_h, r_ref['G'][t_day2], color='black', linewidth=1.8,
                label=r'FTCS reference $\Delta t=15$ s')

    for sname, col in [('FTCS', C_FTCS), ('BTCS', C_BTCS), ('CN', C_CN)]:
        r = test2[(surface, sname, 600.0)]
        if r['blew_up']:
            ax.text(0.50, 0.92,
                    f'{sname} $\\Delta t=600$ s: BLEW UP',
                    transform=ax.transAxes, ha='center', va='top',
                    color=col, fontsize=8.5,
                    bbox=dict(boxstyle='round', facecolor='#ffeeee',
                              edgecolor=col, alpha=0.7))
            continue
        t_day2 = r['times'] >= 86400.0
        t_h = (r['times'][t_day2] - 86400.0) / 3600.0
        ax.plot(t_h, r['G'][t_day2], color=col, linewidth=1.4,
                label=fr'{sname} $\Delta t=600$ s', alpha=0.85)
    ax.set_xlabel('Hour of day')
    ax.set_title(SUBSTRATE_NAMES[surface])
    ax.set_xlim(0, 24); ax.set_xticks([0, 6, 12, 18, 24])
    ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
    ax.legend(loc='lower right', frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r'$G$ (W/m$^2$, +ve = into ground)')

plt.suptitle(r'Day-2 ground heat flux, $\Delta t = 600$ s '
             r'vs FTCS $\Delta t=15$ s reference', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig4_G_dt600.png'), bbox_inches='tight')
plt.show()

### 5g. Figure 5 — Vertical profiles (asphalt road column)

The Ts(z) profile at three local times (06:00, 12:00, 18:00) shows where the schemes diverge. At 06:00 (steady cooling) all schemes match. At 12:00 (peak forcing) BTCS is slightly warmer near surface. At 18:00 (immediately after sunset, the moment of fastest forcing change) BTCS at Δt = 600 s holds the surface ~2 K above the reference, with the deviation concentrated in the top 5 cm — the asphalt layer.

In [ ]:
surface = 'asphalt_road'
target_hours = [6, 12, 18]
target_t = [86400.0 + h * 3600.0 for h in target_hours]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5), sharey=True)
r_ref = test2[(surface, 'FTCS', 15.0)]
z = r_ref['z']

for ax, h, t_target in zip(axes, target_hours, target_t):
    idx_ref = int(np.argmin(np.abs(r_ref['times'] - t_target)))
    ax.plot(r_ref['profiles'][idx_ref] - 273.15, z * 100, color='black',
            linewidth=2.0, label=r'FTCS ref $\Delta t=15$s')
    for sname, col in [('FTCS', C_FTCS), ('BTCS', C_BTCS), ('CN', C_CN)]:
        r = test2[(surface, sname, 600.0)]
        if r['blew_up']:
            continue
        idx = int(np.argmin(np.abs(r['times'] - t_target)))
        ax.plot(r['profiles'][idx] - 273.15, r['z'] * 100,
                color=col, linewidth=1.4,
                label=f'{sname} $\\Delta t=600$s', alpha=0.85)
    ax.invert_yaxis()
    ax.set_xlabel(r'$T_s$ ($^\circ$C)')
    ax.set_title(f'{h:02d}:00 LT')
    ax.set_ylim(50, 0)
    ax.legend(loc='lower right', frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Depth (cm)')

plt.suptitle('Vertical profile, asphalt road column, day 2 - top 50 cm', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig5_profiles_asphalt.png'), bbox_inches='tight')
plt.show()

## 6. Cross-substrate quantitative summary

For each (substrate, scheme, Δt) we compute on day 2:
- **Diurnal amplitude of G:** A_G = (Gmax − Gmin)/2.
- **Amplitude ratio:** A/A_ref relative to FTCS at Δt = 15 s for that substrate.
- **Cross-correlation phase lag** of G(t) vs reference, in minutes.
- **RMS error in surface temperature** Ts⁰ vs reference, in K.

In [ ]:
def diurnal_metrics(test2_results):
    rows = []
    for surface in surfaces:
        ref = test2_results[(surface, 'FTCS', 15.0)]
        if ref['blew_up']:
            continue
        mask_ref = ref['times'] >= 86400.0
        t_ref = ref['times'][mask_ref]
        G_ref = ref['G'][mask_ref]; Ts_ref = ref['Ts0'][mask_ref]
        AG_ref = 0.5 * (np.max(G_ref) - np.min(G_ref))
        for scheme in ['FTCS', 'BTCS', 'CN']:
            for dt in dts:
                r = test2_results[(surface, scheme, dt)]
                if r['blew_up']:
                    rows.append((surface, scheme, dt, np.nan, np.nan, np.nan, np.nan, True))
                    continue
                mask = r['times'] >= 86400.0
                t = r['times'][mask]; G = r['G'][mask]; Ts = r['Ts0'][mask]
                AG = 0.5 * (np.max(G) - np.min(G))
                S = np.trapezoid(G, t)
                Ts_on_ref = np.interp(t_ref, t, Ts)
                rmse = float(np.sqrt(np.mean((Ts_on_ref - Ts_ref)**2)))
                G_on_ref = np.interp(t_ref, t, G)
                lag = compute_lag(G_on_ref, G_ref, t_ref)
                rows.append((surface, scheme, dt, AG, AG/AG_ref, lag, rmse, False))
    return rows


metrics = diurnal_metrics(test2)
print(f"{'Surface':14s} {'Scheme':6s} {'dt(s)':>6} "
      f"{'A_G':>8} {'A/A_ref':>8} {'lag(min)':>9} {'RMSE_T(K)':>10}")
print('-' * 70)
for surf, sch, dt, ag, ratio, lag, rmse, blew in metrics:
    if blew:
        print(f'{surf:14s} {sch:6s} {dt:>6.0f}    BLEW UP')
    else:
        print(f'{surf:14s} {sch:6s} {dt:>6.0f} {ag:>8.1f} {ratio:>8.3f} '
              f'{lag:>9.1f} {rmse:>10.3f}')

## 7. Summary of findings

1. **FTCS bound is set by the most thermally stiff layer.** For asphalt and concrete roof, Δt_crit ≈ 33 s and 17 s respectively — well below operational mesoscale time steps. *This is a structural argument for implicit treatment, regardless of how cheap explicit schemes are.* (Notes #11 Misconception #1.)

2. **CN is roughly twice as accurate as BTCS at every substrate and time step.** Consistent with second-order CN against first-order BTCS, and with the order-of-operations splitting analysis of Notes #10 Section 8 for the SEB-soil coupling.

3. **Substrate-dependence is dominated by layer structure, not admittance.** Bare soil errors are 1/3 of asphalt and 1/5 of concrete-roof at the same scheme and Δt. The mechanism is curvature smearing at sharp λ interfaces (asphalt/aggregate factor of 2; concrete/insulation factor of 37).

4. **The diurnal G amplitude inflation has a specific UHI signature.** The over-amplification is roughly symmetric between day and night, so the daily mean is preserved; the implication is an inflation of the diurnal range of the simulated UHI rather than a bias in its mean. (Notes #11 Misconception #3.)

The natural follow-up is to couple the same diagnostic framework to an offline run of WRF-SLUCM or WRF-TEB urban canopy schemes for a real city's substrate composition with flux-tower-derived forcing.